# KG Consensus Metrics

Calculates Gwet's AC1 between the two experts, plus precision, recall, and F1 for Expert 1, Expert 2, and the final consensus across all available courses.

Labels used: `correct`, `partial`, `wrong`, `missing`. Precision is measured over extracted triples; recall is measured against accepted extracted signal plus validator-added `missingTriples`.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd


# =========================
# 0) CONFIG
# =========================
PROJECT_DIR = Path(os.getenv("PROJECT_DIR", Path.cwd())).expanduser()
CONSENSUS_DIR = Path(os.getenv("CONSENSUS_DIR", PROJECT_DIR / "final-consensus")).expanduser()
if not CONSENSUS_DIR.exists() and PROJECT_DIR.name == "final-consensus":
    CONSENSUS_DIR = PROJECT_DIR

VALIDATION_DIR = CONSENSUS_DIR / "validations"
METRIC_LABELS = ["correct", "partial", "wrong", "missing"]
LABEL_DISPLAY = {
    "correct": "Benar",
    "partial": "Sebagian benar",
    "wrong": "Salah",
    "missing": "Kurang konteks",
}
EXCLUDE_GOLD_PATTERNS = ["mock", "("]


def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(CONSENSUS_DIR))
    except ValueError:
        return path.name


def slug_to_subject(slug: str) -> str:
    return slug.replace("_", " ").replace("-", " ").title()


def discover_courses() -> list[dict]:
    courses = []
    for gold_path in sorted(CONSENSUS_DIR.glob("*_gold_standard.json")):
        if any(pattern in gold_path.name for pattern in EXCLUDE_GOLD_PATTERNS):
            continue
        slug = gold_path.name.removesuffix("_gold_standard.json")
        expert_1_path = VALIDATION_DIR / f"expert-{slug}-4.json"
        expert_2_path = VALIDATION_DIR / f"expert-{slug}-6.json"
        if not expert_1_path.exists() or not expert_2_path.exists():
            print(f"Skipping {slug}: missing expert validation pair")
            continue
        courses.append({
            "subject": slug_to_subject(slug),
            "slug": slug,
            "expert_1_path": expert_1_path,
            "expert_2_path": expert_2_path,
            "gold_path": gold_path,
        })
    return courses


COURSES = discover_courses()
if not COURSES:
    raise FileNotFoundError(f"No complete course metric inputs found in {display_path(CONSENSUS_DIR)}")

print("Courses:")
for course in COURSES:
    print(f"- {course['subject']}: {display_path(course['gold_path'])}")

Courses:
- Biologi: biologi_gold_standard.json
- Fisika: fisika_gold_standard.json
- Kimia: kimia_gold_standard.json


In [2]:
# =========================
# 1) LOAD + HELPERS
# =========================
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def normalize_label(value) -> str:
    text = str(value or "").strip().lower()
    aliases = {
        "ignore": "abaikan",
        "ignored": "abaikan",
        "skip": "abaikan",
        "skipped": "abaikan",
    }
    return aliases.get(text, text)


def metric_label(value):
    label = normalize_label(value)
    return label if label in METRIC_LABELS else None


def normalize_triple_id(value) -> str:
    if value is None:
        return ""
    text = str(value).strip()
    if text.endswith(".0") and text[:-2].isdigit():
        return text[:-2]
    return str(int(text)) if text.isdigit() else text


def expert_rating_map(expert_data: dict) -> dict[str, str]:
    ratings = expert_data.get("ratings", {}) or {}
    out = {}
    for triple_id, raw_label in ratings.items():
        label = metric_label(raw_label)
        if label:
            out[normalize_triple_id(triple_id)] = label
    return out


def consensus_labels(gold_rows: list[dict]) -> list[str]:
    labels = []
    for row in gold_rows:
        if row.get("triple_id") is None:
            continue
        label = metric_label(row.get("final_consensus_label"))
        if label:
            labels.append(label)
    return labels


def accepted_consensus_missing_triples(gold_rows: list[dict]) -> int:
    accepted = 0
    for row in gold_rows:
        if row.get("triple_id") is not None:
            continue
        label = normalize_label(row.get("final_consensus_label"))
        if label not in {"abaikan", "unknown", ""}:
            accepted += 1
    return accepted


def load_course(course: dict) -> dict:
    expert_1_data = load_json(course["expert_1_path"])
    expert_2_data = load_json(course["expert_2_path"])
    gold_rows = load_json(course["gold_path"])
    return {
        **course,
        "expert_1_data": expert_1_data,
        "expert_2_data": expert_2_data,
        "gold_rows": gold_rows,
        "expert_1_ratings": expert_rating_map(expert_1_data),
        "expert_2_ratings": expert_rating_map(expert_2_data),
        "consensus_existing_labels": consensus_labels(gold_rows),
        "expert_1_missing_triples": len(expert_1_data.get("missingTriples", []) or []),
        "expert_2_missing_triples": len(expert_2_data.get("missingTriples", []) or []),
        "consensus_missing_triples": accepted_consensus_missing_triples(gold_rows),
    }


course_data = [load_course(course) for course in COURSES]

summary_rows = []
for course in course_data:
    summary_rows.append({
        "subject": course["subject"],
        "expert_1_rated": len(course["expert_1_ratings"]),
        "expert_2_rated": len(course["expert_2_ratings"]),
        "consensus_existing_labels": len(course["consensus_existing_labels"]),
        "expert_1_missingTriples": course["expert_1_missing_triples"],
        "expert_2_missingTriples": course["expert_2_missing_triples"],
        "consensus_missingTriples": course["consensus_missing_triples"],
    })

summary_table = pd.DataFrame(summary_rows)
display(summary_table)

,subject,expert_1_rated,expert_2_rated,consensus_existing_labels,expert_1_missingTriples,expert_2_missingTriples,consensus_missingTriples
0,Biologi,151,151,151,0,0,0
1,Fisika,240,240,240,15,0,10
2,Kimia,166,166,166,0,29,19


In [3]:
# =========================
# 2) GWET'S AC1
# =========================
def gwets_ac1_for_maps(rater_a: dict[str, str], rater_b: dict[str, str], labels: list[str]) -> dict:
    common_ids = sorted(set(rater_a).intersection(rater_b), key=lambda x: int(x) if x.isdigit() else x)
    pairs = [(rater_a[i], rater_b[i]) for i in common_ids]
    n = len(pairs)
    if n == 0:
        return {
            "items_compared": 0,
            "observed_agreement": 0.0,
            "chance_agreement": 0.0,
            "gwets_ac1": 0.0,
        }

    observed = sum(1 for a, b in pairs if a == b) / n
    total_ratings = 2 * n
    proportions = []
    for label in labels:
        count = sum(1 for a, b in pairs if a == label) + sum(1 for a, b in pairs if b == label)
        proportions.append(count / total_ratings)

    q = len(labels)
    chance = sum(p * (1 - p) for p in proportions) / (q - 1) if q > 1 else 0.0
    ac1 = (observed - chance) / (1 - chance) if chance != 1 else 0.0
    return {
        "items_compared": n,
        "observed_agreement": observed,
        "chance_agreement": chance,
        "gwets_ac1": ac1,
    }


ac1_rows = []
combined_expert_1 = {}
combined_expert_2 = {}
for course in course_data:
    ac1 = gwets_ac1_for_maps(course["expert_1_ratings"], course["expert_2_ratings"], METRIC_LABELS)
    ac1_rows.append({"subject": course["subject"], **ac1})
    for triple_id, label in course["expert_1_ratings"].items():
        combined_expert_1[f"{course['slug']}:{triple_id}"] = label
    for triple_id, label in course["expert_2_ratings"].items():
        combined_expert_2[f"{course['slug']}:{triple_id}"] = label

ac1_rows.append({"subject": "Overall", **gwets_ac1_for_maps(combined_expert_1, combined_expert_2, METRIC_LABELS)})
ac1_table = pd.DataFrame(ac1_rows)

display(ac1_table.style.format({
    "observed_agreement": "{:.3f}",
    "chance_agreement": "{:.3f}",
    "gwets_ac1": "{:.3f}",
}))

,subject,items_compared,observed_agreement,chance_agreement,gwets_ac1
0,Biologi,151,0.934,0.040,0.931
1,Fisika,240,0.771,0.069,0.754
2,Kimia,166,0.651,0.112,0.607
3,Overall,557,0.779,0.076,0.761


In [5]:
# =========================
# 3) PRECISION, RECALL, F1
# =========================
def prf_metrics(labels: list[str], missing_triples_count: int) -> dict:
    clean_labels = [label for label in labels if label in METRIC_LABELS]
    reviewed = len(clean_labels)
    correct = clean_labels.count("correct")
    partial = clean_labels.count("partial")
    wrong = clean_labels.count("wrong")
    missing = clean_labels.count("missing")

    score = correct + partial * 0.5 + missing * 0.25
    precision_base = correct + partial + wrong + missing
    recall_base = correct + partial + missing + missing_triples_count
    precision = score / precision_base if precision_base > 0 else 0.0
    recall = score / recall_base if recall_base > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if precision + recall > 0 else 0.0

    return {
        "reviewed_existing_triples": reviewed,
        "correct": correct,
        "partial": partial,
        "wrong": wrong,
        "missing_label": missing,
        "missingTriples": missing_triples_count,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def add_metric_rows(rows: list[dict], subject: str, actor: str, metrics: dict) -> None:
    rows.append({"subject": subject, "actor": actor, **metrics})


metric_rows = []
overall = {
    "expert_1_labels": [],
    "expert_2_labels": [],
    "consensus_labels": [],
    "expert_1_missing": 0,
    "expert_2_missing": 0,
    "consensus_missing": 0,
}

for course in course_data:
    expert_1_labels = list(course["expert_1_ratings"].values())
    expert_2_labels = list(course["expert_2_ratings"].values())
    consensus_labels_for_course = course["consensus_existing_labels"]

    add_metric_rows(metric_rows, course["subject"], "Expert 1", prf_metrics(expert_1_labels, course["expert_1_missing_triples"]))
    add_metric_rows(metric_rows, course["subject"], "Expert 2", prf_metrics(expert_2_labels, course["expert_2_missing_triples"]))
    add_metric_rows(metric_rows, course["subject"], "Final consensus", prf_metrics(consensus_labels_for_course, course["consensus_missing_triples"]))

    overall["expert_1_labels"].extend(expert_1_labels)
    overall["expert_2_labels"].extend(expert_2_labels)
    overall["consensus_labels"].extend(consensus_labels_for_course)
    overall["expert_1_missing"] += course["expert_1_missing_triples"]
    overall["expert_2_missing"] += course["expert_2_missing_triples"]
    overall["consensus_missing"] += course["consensus_missing_triples"]

add_metric_rows(metric_rows, "Overall", "Expert 1", prf_metrics(overall["expert_1_labels"], overall["expert_1_missing"]))
add_metric_rows(metric_rows, "Overall", "Expert 2", prf_metrics(overall["expert_2_labels"], overall["expert_2_missing"]))
add_metric_rows(metric_rows, "Overall", "Final consensus", prf_metrics(overall["consensus_labels"], overall["consensus_missing"]))

metrics_table = pd.DataFrame(metric_rows)
metrics_table = metrics_table.set_index(["subject", "actor"])

display(metrics_table.style.format({
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
}))

## Notes

- `missing_label` means an existing KG triple was labeled `missing` / `Kurang konteks`.
- `score = correct + 0.5 * partial + 0.25 * missing` — `missing` (`Kurang konteks`) triples earn partial credit.
- Precision = `score / (correct + partial + wrong + missing)`.
- Recall = `score / (correct + partial + missing + missingTriples)`.
- F1 = `2 * precision * recall / (precision + recall)`.
- `missingTriples` means manually added triples proposed by validators. These count as missed ground-truth facts in recall.
- For final consensus, `missingTriples` counts accepted manual additions: rows with `triple_id = null` whose consensus is not `abaikan` or `unknown`.